# NN-PU for 3D — PyTorch port

PyTorch translation of `NNPU_Modified2026_SCNI3D.ipynb` (originally TensorFlow/Keras + TFP on Colab).

**Method.** Neural-network-enriched Reproducing Kernel Particle Method (NN-RKPM) solved by the
Deep Energy Method with SCNI (Stabilized Conforming Nodal Integration):

- An RK background displacement field (coefficients `d*_int`).
- A neural-network enrichment: an MLP of nodal coordinates produces `n_NC` basis functions ζ,
  combined through enrichment coefficients `d*_NN` and gated by a spatial activation mask.
- Smoothed strains from precomputed sparse derivative operators `P1/P2/P3`; isotropic linear-elastic
  bimaterial stresses.
- Loss = total potential energy = `scaling · Σ(strain_energy_density · WT)`.
- Three stages: Adam (RK only) → Adam (RK + NN + network) → L-BFGS (all).

This notebook reuses the tested implementation in `nnpu_torch.py`. Epoch counts are reduced for
interactive use; the production counts (20000 / 30000 / 2000) live in `nnpu_torch.py` and the SLURM
scripts. Select the project's `.venv` as the kernel.

## Imports & device

In [ ]:
import os, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from nnpu_torch import NNRK   # read_dense, read_sparse, EnrichmentMLP also available

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Setup & data (notebook cells 4, 6, 7)

In [ ]:
INPUT_DIR = "Input_3Dbimat_Node396_Cell2662"
nHL, nNR, actv = 5, [40, 40, 40, 40, 5], ["elu"] * 5

prob = NNRK(INPUT_DIR, device, torch.float64, nHL, nNR, actv, gy_ebc=0.01, seed=42)

print("n_cell, n_smooth, n_node =", prob.n_cell, prob.n_smooth, prob.n_node)
print("n_ebc = ", (prob.n_ebc_x, prob.n_ebc_y, prob.n_ebc_z), " n_NC =", prob.n_NC)
print("mat_energy_scaling =", prob.mat_energy_scaling, " d_scaling =", prob.d_scaling)
print("activated nodes:", int(prob.is_activated.sum().item()))

## Orthonormalize the NN basis — diagnostic (notebook cell 12)

Projects the raw MLP basis onto the RK-orthogonal complement; the maximum RK inner product should be ~0,
validating the sparse-transpose / `linalg.solve` / sparse-matmul path.

In [ ]:
print("max RK inner product after orthonormalization:", prob.orthogonality_check())

## Plot helper (z = 0 slice) — mirrors `Plot_3D_Slice`

In [ ]:
def slice_plot(values, title="", z_slice=0.0, tol=1e-3):
    X = prob._np(prob.X_cell); v = np.asarray(values).reshape(-1)
    m = np.abs(X[:, 2] - z_slice) < tol
    plt.figure(figsize=(4, 4))
    plt.tripcolor(X[m, 0], X[m, 1], v[m], cmap="jet"); plt.colorbar()
    plt.title(title); plt.axis("equal"); plt.show()

# activation mask over the RK nodes
Xn = prob._np(prob.X_node); a = prob._np(prob.is_activated).reshape(-1)
plt.figure(figsize=(5, 4)); plt.scatter(Xn[:, 0], Xn[:, 1], c=a, cmap="Reds", s=12)
plt.colorbar(); plt.axis("equal"); plt.title("activated nodes (NN enrichment region)"); plt.show()

## Stage 1 — Adam, RK background only (notebook cells 22–23)

In [ ]:
EP_RK = 3000          # production: 20000
h_rk = prob.train_adam_rk(EP_RK, lr=1e-3)

plt.figure(figsize=(8, 4)); plt.semilogy(h_rk); plt.grid(True, which="both")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Adam (RK) convergence"); plt.show()

with torch.no_grad():
    ux, uy, uz = prob.rk_only(prob.SHP_cell)
    e = prob.strain_rk(); s = prob.stress(*e)
slice_plot(prob._np(ux), "ux (RK)")
slice_plot(prob._np(uy), "uy (RK)")
slice_plot(prob._np(s[1]), "s22 (RK)")

## Stage 2 — Adam, RK + NN enrichment (notebook cells 24–28)

In [ ]:
EP_NNRK = 5000        # production: 30000
h_nnrk = prob.train_adam_nnrk(EP_NNRK, lr=1e-4)

plt.figure(figsize=(8, 4)); plt.semilogy(h_nnrk); plt.grid(True, which="both")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Adam (NN-RK) convergence"); plt.show()

with torch.no_grad():
    ux, uy, uz = prob.total_approx(prob.X_cell, prob.SHP_cell)
    _, uy_nn, _, _ = prob.total_approx_nn(prob.X_cell, prob.SHP_cell)
    e = prob.strain_nnrk(); s = prob.stress(*e)
slice_plot(prob._np(uy), "uy (total)")
slice_plot(prob._np(uy_nn), "uy (NN enrichment part)")
slice_plot(prob._np(s[1]), "s22 (total)")

## Stage 3 — L-BFGS, all parameters (notebook cells 30–32)

In [ ]:
h_lbfgs = prob.train_lbfgs(max_iter=500)   # production: 2000

plt.figure(figsize=(8, 4)); plt.semilogy(h_lbfgs); plt.grid(True, which="both")
plt.xlabel("function evaluation"); plt.ylabel("loss"); plt.title("L-BFGS convergence"); plt.show()

with torch.no_grad():
    ux, uy, uz = prob.total_approx(prob.X_cell, prob.SHP_cell)
    e = prob.strain_nnrk(); s = prob.stress(*e)
slice_plot(prob._np(uy), "uy (final)")
slice_plot(prob._np(s[1]), "s22 (final)")
print("final total potential energy:", float(prob.energy_nnrk()))

## Save solutions, enrichment functions + Jacobians, network weights (cells 27–38)

In [ ]:
OUT = os.path.join(INPUT_DIR, "Results_40NR_4hidden_5bases")
os.makedirs(os.path.join(OUT, "Final"), exist_ok=True)

prob.write_solution(os.path.join(OUT, "results_nnrk_LBFGS.txt"))
prob.save_enrichment_jacobian(OUT, "LBFGS")          # ζ and dζ/dx at cell/smooth/node points
prob.save_layer_weights(os.path.join(OUT, "Final"))
prob.save_checkpoint(os.path.join(OUT, "Final", "LBFGS"), tag="NNRK")
print("artifacts written to", OUT)

## Enrichment basis functions (notebook cell 39)

In [ ]:
with torch.no_grad():
    zeta = prob.model(prob.X_cell)
for k in range(prob.n_NC):
    slice_plot(prob._np(zeta[:, k]), f"enrichment basis {k}")